# Fire-weather indices over CONUS from ERA5

This notebook walks the `climate_indices.fire` family end to end on a
public ERA5 subset:

```text
prepared ERA5 subset (daily surface fields + daily level profiles)
     ->  KBDI, CFFWIS (FFMC/DMC/DC/ISI/BUI/FWI/DSR), Fosberg FFWI, HDW
prepared 30-year monthly baseline (1991-2020)
     ->  SPI-3 and EDDI-3 for the 2020 fire season
     ->  seasonal means, 90th/95th percentiles, and one documented case day
```

**Scientific question.** Which parts of the CONUS domain were simultaneously
dry (deep fuel and duff dryness) and fire-weather dangerous during the 2020
fire season?

**Expected outputs.** Inline maps of seasonal mean FWI, end-of-season DC,
percentile-ranked KBDI and Fosberg FFWI, the 7 September 2020 HDW, and a
four-panel drought-to-fire-weather view placing SPI-3 and EDDI-3 beside DC
and KBDI. Numbers are computed on a 1.5 degree coarse grid over 25-50 N,
235-295 E; the section *What this demo does not claim* states every
approximation.


## Terms used throughout

- **KBDI (Keetch-Byram Drought Index)**: daily recursive measure of cumulative
  moisture deficiency in deep duff and upper soil layers, on a 0-800 scale of
  hundredths of an inch (0-203.2 mm metric), from precipitation, daily maximum
  temperature, and **mean annual precipitation**. Moisture loss reverses only
  through net rain: consecutive rainy days form one wet spell, and only rain
  above its first 5.08 mm reduces the index.
- **CFFWIS (Canadian Forest Fire Weather Index System)**: the three moisture
  codes **FFMC**, **DMC**, and **DC** (fine fuel, duff, and deep compact
  organic layers), plus the behavior indices **ISI** (initial spread),
  **BUI** (buildup), **FWI** (the headline index), and **DSR** (daily severity
  rating). CFFWIS is defined on **noon local-standard-time** temperature,
  relative humidity, and 10 m wind, with 24-hour rain.
- **FFWI (Fosberg Fire Weather Index)**: dimensionless, weather-only,
  elementwise index from temperature, relative humidity, and wind speed; no
  state, no rain term.
- **HDW (Hot-Dry-Windy Index)**: vapor pressure deficit times wind speed,
  maximized over the levels in the lowest **500 m above ground level** of a
  vertical profile, in hPa m s-1.
- **Timescale-free caveat**: none of these indices predicts whether, where, or
  when a fire starts or spreads; they describe weather and fuel-dryness
  context only.


## Environment and data setup

From the repository root:

```bash
uv sync --group dev
uv run --group dev scripts/prepare_fire_demo_inputs.py   # one-time, cached
uv run jupyter lab notebooks/fire_weather_demo.ipynb
```

The preparation script reads the anonymously accessible ARCO-ERA5 store on
Google Cloud Storage, subsets the CONUS box on the coarse 1.5 degree grid, and
writes daily NetCDF files under `data/fire-demo/` (git-ignored). The first run
downloads about a gigabyte of compressed chunks and takes on the order of ten
minutes; per-variable downloads are cached, so a second run needs no network.

Two prepared files drive everything below:

| File | Window | Contents |
| --- | --- | --- |
| `surface_daily_2018_2020.nc` | 2018-2020, daily | `tmean_c`, `tmax_c`, `tmin_c`, `precip_mm`, `wind_speed_ms` |
| `levels_daily_2020_season.nc` | 2020-03-01 to 2020-10-31, daily | `temperature_c`, `relative_humidity_percent`, `wind_speed_ms`, `height_agl_m` on 13 pressure levels, plus `surface_relative_humidity_percent` |

The `CLIMATE_INDICES_FIRE_DEMO_DATA` environment variable overrides the data
directory; the default `../data/fire-demo` resolves against the kernel's
working directory, which is `notebooks/` for the documented launch command.


In [ ]:
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from climate_indices import fire

data_dir = Path(os.environ.get("CLIMATE_INDICES_FIRE_DEMO_DATA", "../data/fire-demo"))
surface_path = data_dir / "surface_daily_2018_2020.nc"
levels_path = data_dir / "levels_daily_2020_season.nc"
baseline_path = data_dir / "surface_monthly_1991_2020.nc"
for path in (surface_path, levels_path, baseline_path, data_dir / "manifest.json"):
    if not path.exists():
        raise FileNotFoundError(
            f"Prepared input not found: {path}. "
            "Generate it with: uv run --group dev scripts/prepare_fire_demo_inputs.py"
        )

# the documented fire season, and the percentile levels the maps report
season_start, season_end = "2020-03-01", "2020-10-31"
percentile_levels = (0.90, 0.95)
case_day = "2020-09-07"  # Labor Day 2020 wind event

# open lazily: the fire adapters constrain which dimension may span chunks,
# and Dask keeps those constraints visible instead of hiding them in memory
surface = xr.open_dataset(surface_path, chunks={})
levels = xr.open_dataset(levels_path, chunks={})
baseline = xr.open_dataset(baseline_path, chunks={})
surface


In [ ]:
import hashlib
import json

def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

manifest = json.loads((data_dir / "manifest.json").read_text())

# an interrupted regeneration can leave one prepared file paired with a stale
# counterpart; the manifest's recorded hashes catch that before any index runs
for prepared_path in (surface_path, levels_path, baseline_path):
    recorded_sha256 = manifest["artifacts"][prepared_path.name]["sha256"]
    actual_sha256 = _sha256(prepared_path)
    if actual_sha256 != recorded_sha256:
        raise RuntimeError(
            f"{prepared_path.name} does not match the sha256 recorded in manifest.json "
            "(a previous regeneration may have been interrupted). Regenerate with: "
            "uv run --group dev scripts/prepare_fire_demo_inputs.py"
        )

print(manifest["source_description"])
print(f"domain: {manifest['domain']}")
print(f"demonstration years: {manifest['surface_years']}")
print(f"calibration baseline: {manifest['baseline_years']}")
print(f"season: {manifest['season']}")
for note in manifest["approximations"]:
    print("-", note)


## KBDI: cumulative moisture deficiency over three years

KBDI's climate factor scales drying with the **mean annual precipitation** of
each cell. ERA5 publishes daily fields but the fire recurrences want a
climatology; the adapter requires at least 30 years of record before it will
infer one, so this demo derives the mean annual total from the three prepared
years and passes it explicitly. `spin_up=365` computes the first year without
returning it, which lets the recursive state settle before the reported
record starts.


In [ ]:
started = time.perf_counter()
mean_annual_precipitation = surface["precip_mm"].groupby("time.year").sum("time").mean("year")

kbdi = fire.kbdi(
    precipitation=surface["precip_mm"],
    maximum_temperature=surface["tmax_c"],
    mean_annual_precipitation=mean_annual_precipitation,
    spin_up=365,
)
kbdi.load()
kbdi_seconds = time.perf_counter() - started
print(f"KBDI computed in {kbdi_seconds:.2f} s on {kbdi.sizes['latitude']}x{kbdi.sizes['longitude']} cells")
kbdi


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
mean_annual_precipitation.plot(ax=axes[0], cmap="YlGnBu", cbar_kwargs={"label": "mm"})
axes[0].set_title("Mean annual precipitation (3-year mean)")
kbdi.isel(time=-1).plot(ax=axes[1], cmap="YlOrRd", cbar_kwargs={"label": "KBDI (mm, 0-203.2)"})
axes[1].set_title(f"KBDI at {np.datetime_as_string(kbdi.time.values[-1], unit='D')}")
plt.tight_layout()


## CFFWIS: the Canadian Forest Fire Weather Index System

`fire.cffwis()` runs the three moisture codes in one daily pass and derives
ISI, BUI, FWI, and DSR from them, returning an `xarray.Dataset` with one
variable per selected output. It infers the month from the datetime `time`
coordinate and the latitude from the `latitude` coordinate, and it requires a
**single Dask chunk along `time`** because the recurrences thread state across
days.

The prepared daily summaries are timestamped 12:00 and stand in for the noon
local-standard-time observations the system is defined on; that approximation
and its consequences are restated in *What this demo does not claim*.


In [ ]:
season = surface.sel(time=slice(season_start, season_end))
surface_relative_humidity = levels["surface_relative_humidity_percent"]

started = time.perf_counter()
cffwis = fire.cffwis(
    temperature_celsius=season["tmean_c"],
    relative_humidity_percent=surface_relative_humidity,
    wind_speed_meters_per_second=season["wind_speed_ms"],
    precipitation_mm=season["precip_mm"],
    outputs=("ffmc", "dmc", "dc", "isi", "bui", "fwi", "dsr"),
)
cffwis.load()
cffwis_seconds = time.perf_counter() - started
print(f"CFFWIS computed in {cffwis_seconds:.2f} s")
cffwis


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
cffwis["fwi"].mean("time").plot(ax=axes[0], cmap="YlOrRd", cbar_kwargs={"label": "FWI"})
axes[0].set_title("Seasonal mean FWI")
cffwis["fwi"].quantile(0.95, dim="time").plot(ax=axes[1], cmap="YlOrRd", cbar_kwargs={"label": "FWI"})
axes[1].set_title("95th percentile FWI")
cffwis["dc"].isel(time=-1).plot(ax=axes[2], cmap="YlOrBr", cbar_kwargs={"label": "DC"})
axes[2].set_title("DC at the end of the season")
plt.tight_layout()


## Fosberg FFWI: the elementwise member of the family

The Fosberg Fire Weather Index is stateless and elementwise. The xarray
adapters in this package are added function by function, and the Fosberg
index remains available through the stable NumPy layer only, so the notebook
wraps `fire.fosberg_ffwi` with `xr.apply_ufunc` to keep the labelled
dimensions and the Dask graph. The calculation stays lazy until it is
displayed or reduced.


In [ ]:
started = time.perf_counter()
fosberg = xr.apply_ufunc(
    fire.fosberg_ffwi,
    season["tmean_c"],
    surface_relative_humidity,
    season["wind_speed_ms"],
    dask="parallelized",
    output_dtypes=[np.float64],
)
fosberg.load()
fosberg_seconds = time.perf_counter() - started
fosberg.name = "fosberg_ffwi"
fosberg.attrs["units"] = "1"
fosberg.attrs["long_name"] = "Fosberg Fire Weather Index"
print(f"Fosberg FFWI computed in {fosberg_seconds:.2f} s")

percentile_90, percentile_95 = percentile_levels
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fosberg.quantile(percentile_90, dim="time").plot(
    ax=axes[0], cmap="YlOrRd", cbar_kwargs={"label": "FFWI"}
)
axes[0].set_title("90th percentile Fosberg FFWI")
fosberg.quantile(percentile_95, dim="time").plot(
    ax=axes[1], cmap="YlOrRd", cbar_kwargs={"label": "FFWI"}
)
axes[1].set_title("95th percentile Fosberg FFWI")
plt.tight_layout()


## HDW: a vertical-profile index

HDW is the only index here that consumes a vertical dimension: the maximum of
vapor pressure deficit times wind speed over the levels in the lowest 500 m
above ground. The prepared level file carries temperature, relative humidity,
wind speed, and height above ground on 13 pressure levels, so the call needs
no pressure-to-height conversion. Columns whose levels all sit outside the
500 m layer return NaN. The empty-column warning is suppressed for
Dask-backed input, which is how the prepared files are opened here.

Note the formulation: `fire.hot_dry_windy()` implements the per-level product
contracted in this package's design, which never exceeds the published Srock
et al. (2018) variant; comparing absolute values against published HDW maps
requires that caveat.


In [ ]:
started = time.perf_counter()
hdw = fire.hot_dry_windy(
    levels["temperature_c"],
    levels["relative_humidity_percent"],
    levels["wind_speed_ms"],
    levels["height_agl_m"],
)
hdw.load()
hdw_seconds = time.perf_counter() - started
print(f"HDW computed in {hdw_seconds:.2f} s")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
hdw.sel(time=case_day).plot(ax=axes[0], cmap="magma", cbar_kwargs={"label": "hPa m s-1"})
axes[0].set_title(f"HDW on {case_day}")
hdw.max("time").plot(ax=axes[1], cmap="magma", cbar_kwargs={"label": "hPa m s-1"})
axes[1].set_title("HDW seasonal maximum")
plt.tight_layout()


## The drought-to-fire-weather chain

The physical story this family supports is a chain: Antecedent dryness (deep
fuel and duff moisture: DC, KBDI) lowers the moisture the fire-weather
indices have to work against, and the fire-weather indices describe the wind
and vapor-pressure-deficit side. The panels below place the standardized
drought indices for September 2020, SPI-3 and EDDI-3, beside end-of-season DC
and 95th-percentile KBDI.


## Standardized drought indices on a 30-year baseline

SPI and EDDI rank each month against the same calendar month across a
calibration record, and the xarray adapters refuse to fit a distribution from
fewer than 30 years of non-NaN data. The three-year record above cannot supply
that, so `scripts/prepare_fire_demo_inputs.py` also prepares a monthly
baseline over the WMO 1991-2020 normal period on the same grid.

EDDI needs potential evapotranspiration, not temperature alone. This demo uses
the Thornthwaite method (`climate_indices.pet_thornthwaite`), the package's
monthly PET form; it is a temperature-only estimate and omits the humidity,
wind, and radiation terms of a Penman-Monteith reference ET. The calibration
window contains the 2020 event year, so the September 2020 ranks are
within-sample and demonstrate the API rather than independently verifying the
event.

September 2020 ranks a three-month accumulation: SPI-3 over July-September
precipitation and EDDI-3 over July-September demand.


In [ ]:
from climate_indices import eddi, pet_thornthwaite, spi
from climate_indices.indices import Distribution

started = time.perf_counter()
pet = pet_thornthwaite(baseline["tmean_c"], baseline["latitude"])
pet.load()
pet.name = "pet"
pet_seconds = time.perf_counter() - started
print(f"Thornthwaite PET over the 30-year baseline in {pet_seconds:.2f} s")
pet


In [ ]:
started = time.perf_counter()
spi_3 = spi(
    baseline["precip_mm"],
    scale=3,
    distribution=Distribution.gamma,
    calibration_year_initial=1991,
    calibration_year_final=2020,
)
eddi_3 = eddi(
    pet,
    scale=3,
    calibration_year_initial=1991,
    calibration_year_final=2020,
)
spi_3.load()
eddi_3.load()
spi_3.name = "spi_3"
eddi_3.name = "eddi_3"
standardized_seconds = time.perf_counter() - started
print(f"SPI-3 and EDDI-3 over the 30-year baseline in {standardized_seconds:.2f} s")

# the September stamps sit at 12:00, so a partial-string selection keeps a
# length-one time dimension; squeeze it away for a map
spi_september = spi_3.sel(time="2020-09-01").squeeze("time", drop=True)
eddi_september = eddi_3.sel(time="2020-09-01").squeeze("time", drop=True)
spi_september


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
spi_september.plot(
    ax=axes[0, 0], cmap="BrBG", vmin=-3.0, vmax=3.0, cbar_kwargs={"label": "SPI-3"}
)
axes[0, 0].set_title("SPI-3, September 2020")
eddi_september.plot(
    ax=axes[0, 1], cmap="RdBu_r", vmin=-3.0, vmax=3.0, cbar_kwargs={"label": "EDDI-3"}
)
axes[0, 1].set_title("EDDI-3, September 2020")
cffwis["dc"].isel(time=-1).plot(ax=axes[1, 0], cmap="YlOrBr", cbar_kwargs={"label": "DC"})
axes[1, 0].set_title("DC at the end of the season")
kbdi.sel(time=slice(season_start, season_end)).quantile(0.95, dim="time").plot(
    ax=axes[1, 1], cmap="YlOrRd", cbar_kwargs={"label": "KBDI"}
)
axes[1, 1].set_title("95th percentile KBDI")
plt.tight_layout()


## Timing and chunking notes

The recurrences (`KBDI`, `CFFWIS`) require the `time` dimension to be a single
Dask chunk; `HDW` requires its `level` dimension to be a single chunk and
leaves every other dimension free. The cells above run on the default
scheduler from Dask-backed single-chunk arrays, so the table below is
dominated by the vectorized NumPy cores, not by Dask scheduling. On this 17x40 grid the
whole family is seconds; the chunking contract matters when the same code runs
on operational grids, where one cell column - not one cell - becomes the unit
of state.

Memory: the prepared files total a few megabytes and every array above is
small enough to fit in memory many times over. Streaming the same computation
across a full-resolution ERA5 grid would need Dask, and the adapter
constraints are what make that possible without changing the call.


In [ ]:
timings = {
    "KBDI (3 years)": kbdi_seconds,
    "CFFWIS (season, 7 outputs)": cffwis_seconds,
    "Fosberg FFWI (season)": fosberg_seconds,
    "HDW (season, 13 levels)": hdw_seconds,
    "Thornthwaite PET (30 years)": pet_seconds,
    "SPI-3 and EDDI-3 (30 years)": standardized_seconds,
}
for name, seconds in timings.items():
    print(f"{name:32s} {seconds:6.2f} s")

# the chunk contract each adapter enforces: one time chunk for the recurrences,
# one level chunk for HDW, and no constraint on the other axes
print("surface chunks: ", {dim: surface.chunksizes[dim] for dim in surface["tmean_c"].dims})
print("levels chunks:  ", {dim: levels.chunksizes[dim] for dim in levels["temperature_c"].dims})


## Qualitative comparison with GFWED

NASA GISS's [Global Fire Weather Database](https://data.giss.nasa.gov/impacts/gfwed/)
(GFWED) is built on the CFFWIS daily severity rating and provides a useful
conceptual comparison target. This notebook makes **no** quantitative
agreement claim: the demo runs on a coarse 1.5 degree subset with daily-mean
inputs standing in for noon observations, a derived relative humidity, and a
three-year precipitation baseline for KBDI, all of which shift absolute
values relative to an operational product. The comparison is at the pattern
level only: both approaches expect the largest fire-weather potential in the
interior West during the dry season.


## What this demo does not claim

- **Coarse resolution.** 1.5 degree grid cells, 25-50 N. Not an operational
  grid. The box is not masked to the United States: it also covers northern
  Mexico and southern Canada.
- **Derived humidity.** ERA5's 2 m dewpoint is absent from the coarse store,
  so relative humidity comes from specific humidity and temperature at the
  lowest pressure level above the surface, via the FAO-56 saturation vapor
  pressure used by `pm_eto`. Over high terrain that level can sit hundreds of
  meters above the surface.
- **Midday daily summaries.** CFFWIS is defined on noon local-standard-time
  observations; the prepared inputs are daily means/maxima/sums timestamped
  12:00. Diurnal swings in temperature and humidity are smoothed away.
- **Three-year KBDI baseline.** Mean annual precipitation is a three-year
  mean, not a climate normal; the adapter refuses to infer one from records
  shorter than 30 years.
- **Temperature-only PET.** EDDI uses Thornthwaite PET over the monthly
  baseline: a temperature-only estimate that omits the humidity, wind, and
  radiation terms of a Penman-Monteith reference ET, and a monthly
  accumulation that smooths the sub-monthly demand spikes 14-day EDDI
  resolves.
- **In-sample calibration.** SPI-3 and EDDI-3 rank against the WMO 1991-2020
  normal period, which contains the 2020 event year, so the September 2020
  values are not an independent verification.
- **HDW formulation.** The per-level VPD x wind product implemented here is
  the package's contracted variant, which never exceeds the published
  Srock et al. (2018) surface-adjusted formulation.
- **Cold-started CFFWIS.** The moisture codes start from the literature seeds
  (FFMC 85, DMC 6, DC 15) on 1 March 2020: the 2018-2019 surface record could
  not spin them up because it carries no derived relative humidity. DC
  therefore reports one season's drying, not multi-year drought memory.
- **No validation against observations.** Nothing here is checked against an
  operational FWI product, GFWED, or fire occurrence records; that is the
  subject of the fire validation work in `VALIDATION.md`.


## Where to go next

- `docs/wildfire_applications.md`: the fire family and the drought-to-fire
  chain, with the caveats behind each index.
- `docs/design/fire-subsystem.md`: the API tiers, the state contract, and the
  scope boundary.
- `src/climate_indices/CONTEXT.md`: the vocabulary used above.
- A full-resolution version of this workflow is tracked as a follow-up to the
  fire subsystem work.
